# Hand-On Week 4: 

## ASCII-based Text-Representations

In this notebook, we build a simple language detector (English vs. Spanish) by converting characters into their ASCII and Binary representations.

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)

In [2]:
# Load the data into Python
texts, labels = [], []
with open("../../data/lang_detection.txt", "r", encoding="utf-8") as f:
    for line in f:
        sentence, label = line.strip().split("\t")
        texts.append(sentence)
        labels.append(label)
        
y = np.array(labels)

In [3]:
texts[:4]

['hello world', 'how are you', 'this is great', 'good morning']

In [4]:
set(labels)

{'de', 'en', 'es'}

In [5]:
for char in "hello":
    print("Character: ", char, "\t ASCII: ", ord(char), "\t Bit Vektor: ", format(ord(char), '08b'))

Character:  h 	 ASCII:  104 	 Bit Vektor:  01101000
Character:  e 	 ASCII:  101 	 Bit Vektor:  01100101
Character:  l 	 ASCII:  108 	 Bit Vektor:  01101100
Character:  l 	 ASCII:  108 	 Bit Vektor:  01101100
Character:  o 	 ASCII:  111 	 Bit Vektor:  01101111


In the cell below, implement a function that takes a text phrase as input and returns a numerical vector which the Logistic Regression function can process. 

**HINT**: Feature vectors must be of fixed size, as there is a fixed size of parameters in the model. A model with two parameters $w_1$ and $w_2$ (and an optional bias parameter $b$) always needs two input features!

In [6]:
[int(x) for x in list(format(ord(char), '08b'))]


[0, 1, 1, 0, 1, 1, 1, 1]

In [7]:
def text_to_binary(text, max_len=20):
    """Extend this function to generate a feature vector from text as input for the classifier"""
    # Pad or truncate text
    text = text.ljust(max_len)[:max_len]
    # TODO: Come up with a way to generate a numerical feature vector for our text classifier
    binary_vector = []
    for char in text:
        # Convert char to 8-bit binary string, then to list of ints
        bits = list(format(ord(char), '08b'))
        # TODO: Come up with a way to generate a numerical feature vector for our text classifier
        binary_vector.extend([int(b) for b in bits])
        
    return binary_vector
    
X = np.array([text_to_binary(t, max_len=20) for t in texts])

In [8]:
X[:4].shape

(4, 160)

Train the logistic regression using sklearns ```.fit()``` method. Test your classifier on a hold-out dataset.

**HINT**: Use ```train_test_split``` to generate a train and test dataset

In [9]:
# Split and Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

# Quick Test
test_phrase = "como estas"
vec = np.array([text_to_binary(test_phrase, max_len=20)])
prediction = model.predict(vec)

print(f"Phrase: '{test_phrase}' | Prediction: {prediction[0]}")

Phrase: 'como estas' | Prediction: es


In [10]:
prediction

array(['es'], dtype='<U2')

## Evaluation

Compute the accuracy of the trained classifier. 

In [11]:
model.predict(X_test) == y_test

array([False, False,  True, False, False, False,  True,  True,  True,
        True, False,  True,  True,  True,  True, False,  True, False,
        True, False,  True,  True,  True, False, False,  True, False,
        True,  True,  True])

In [12]:
float((model.predict(X_test) == y_test).sum() / len(y_test)) * 100

60.0

# Problems with this approach

In [13]:
import numpy as np
from scipy.spatial.distance import cosine

# Helper function to compare two binary vectors
def compare_vectors(word1, word2, max_len=10):
    vec1 = np.array(text_to_binary(word1, max_len))
    vec2 = np.array(text_to_binary(word2, max_len))
    
    # Calculate cosine distance
    distance = cosine(vec1, vec2)
    similarity = (1 - distance) * 100
    
    print(f"'{word1}' vs '{word2}' -> Binary Similarity: {similarity:.2f}%")

In [14]:
compare_vectors(
    "This is the first sentence without meaning.", 
    "This is the first sentence without meaning. The interesting information is in this sentence."
) 

'This is the first sentence without meaning.' vs 'This is the first sentence without meaning. The interesting information is in this sentence.' -> Binary Similarity: 100.00%


### Semantic Blindness
The model looks at spelling, not meaning. Words that mean the exact same thing look completely different to the model, while words that rhyme but mean different things look identical.

In [15]:
# Synonyms: Mean the same thing, but spelled differently
compare_vectors("dog", "pup") 
compare_vectors("happy", "joyful")

# Rhymes: Mean completely different things, but spelled similarly
compare_vectors("dog", "bog")

# Conclusion: The model thinks "dog" is much closer to a swamp ("bog") 
# than it is to a baby dog ("pup").

'dog' vs 'pup' -> Binary Similarity: 77.15%
'happy' vs 'joyful' -> Binary Similarity: 75.38%
'dog' vs 'bog' -> Binary Similarity: 95.24%


### Positional Fragility
Because our features are locked to specific positions (e.g., bits 0-7 are the first character), shifting the text by just one space ruins the entire vector.

In [16]:
word = "hello"
word_shifted = " hello" # Added one space at the beginning

compare_vectors(word, word_shifted)

# Let's see what the model actually predicts for the shifted word
vec_normal = np.array([text_to_binary(word)])
vec_shifted = np.array([text_to_binary(word_shifted)])

'hello' vs ' hello' -> Binary Similarity: 73.08%


### Orthgraphic Fragility
Because our features are locked to specific positions (e.g., bits 0-7 are the first character), shifting the text by just one space ruins the entire vector.

In [17]:
word = "joyful"
word_shifted = "jyful" # Added one space at the beginning

compare_vectors(word, word_shifted, max_len=6)

# Let's see what the model actually predicts for the shifted word
vec_normal = np.array([text_to_binary(word)])
vec_shifted = np.array([text_to_binary(word_shifted)])

'joyful' vs 'jyful' -> Binary Similarity: 66.99%
